# Python Bridge for 05a

## A Short Code Refresher

Use this optional notebook alongside 05a for a closer look at the Python mechanics of nonlinear dimensionality reduction. Plan for about 20 minutes, plus the first UMAP run's compilation time. It uses a small subset of the digit images bundled with scikit-learn, so no data download is needed.

The lecture explains how representations and neighborhood choices affect a map. This companion follows five code patterns:

1. connect an image to its numerical feature row;
2. separate recorded labels from model inputs;
3. create a model and obtain two-dimensional coordinates;
4. attach row labels and plot the coordinates; and
5. distinguish a fitted object from its returned array.

## References

The [digits dataset reference](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) describes the images and recorded labels. The [t-SNE reference](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) and [UMAP API guide](https://umap-learn.readthedocs.io/en/latest/api.html) describe the settings and methods used below.

## Load Images and Feature Rows

Each image has 8 rows and 8 columns of pixel intensities. The dataset also stores those same 64 values as one feature row, matching the representation used in 05a. We retain the first 200 images to keep this code exercise small.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from umap import UMAP

In [ ]:
digits = load_digits()
features = digits.data[:200]
images = digits.images[:200]
recorded_digits = digits.target[:200]
row_ids = pd.Index(range(len(features)), name="Image row")

print("image-array shape:", images.shape)
print("feature-array shape:", features.shape)
print("recorded-label shape:", recorded_digits.shape)

The shapes are `(200, 8, 8)`, `(200, 64)`, and `(200,)`. The first axis always identifies the image. A recorded digit identifies what was written; it is not one of the 64 input features.

## Inspect One Representation

`features[0]` selects the first image's one-dimensional feature row. `.reshape(8, 8)` restores its known grid shape. No pixel values are discarded by this change in shape.

In [ ]:
first_grid = features[0].reshape(8, 8)

print("feature row:", features[0].shape)
print("restored grid:", first_grid.shape)
print("matches the stored image:", np.array_equal(first_grid, images[0]))
pd.DataFrame(first_grid.astype(int))

The equality check returns `True`. The model input remains the two-dimensional `(200, 64)` array: one row per image and one column per pixel position. We use the original pixel scale, as in the lecture. Every column measures the same kind of intensity; standardizing pixels separately would change the distance question.

## Fit t-SNE and Retain Its Coordinates

`TSNE(...)` creates the model object. `n_components=2` requests two output coordinates. `perplexity=15` sets the neighborhood scale for this small example and must be less than the number of input observations. `init="pca"` chooses the starting layout, while `random_state=7130` fixes the random choices. The model still compares the original 64-feature rows.

In [ ]:
tsne_model = TSNE(
    n_components=2,
    perplexity=15,
    init="pca",
    learning_rate="auto",
    max_iter=1000,
    random_state=7130,
)
tsne_coordinates = tsne_model.fit_transform(features)

print("returned-coordinate shape:", tsne_coordinates.shape)
print("stored-coordinate shape:", tsne_model.embedding_.shape)
print(
    "returned and stored coordinates agree:", np.allclose(tsne_coordinates, tsne_model.embedding_)
)

`tsne_model` is the fitted object; `tsne_coordinates` is a `(200, 2)` array. Each output row corresponds to the same input image. The two columns are map coordinates, not pixel measurements or PCA loadings. `max_iter` bounds the layout optimization; it does not request 1,000 separate maps.

## Restore Labels for Inspection

The returned array has no pandas row labels. Attach the original row IDs before adding the recorded digits used for interpretation.

In [ ]:
tsne_table = pd.DataFrame(tsne_coordinates, index=row_ids, columns=["Map 1", "Map 2"])
tsne_table["Recorded digit"] = recorded_digits

tsne_table.head().round(3)

There are two calculated coordinates and one recorded label per row. The digit labels never entered `fit_transform`. Adding them afterward lets us inspect whether nearby images tend to depict the same digit without using those labels to construct the map.

## Fit UMAP to the Same Input Rows

The UMAP call has a similar shape. `n_neighbors=10` sets the input neighborhood size, and `min_dist=0.1` controls how tightly points can be arranged in the layout. It is not a guaranteed minimum separation between every pair. The fixed seed and `n_jobs=1` support repeatable execution. UMAP's first call can take longer while its numerical routines compile.

In [ ]:
umap_model = UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.1,
    metric="euclidean",
    random_state=7130,
    n_jobs=1,
)
umap_coordinates = umap_model.fit_transform(features)
umap_table = pd.DataFrame(umap_coordinates, index=row_ids, columns=["Map 1", "Map 2"])

print("coordinate shape:", umap_coordinates.shape)
print("input-neighborhood graph shape:", umap_model.graph_.shape)
umap_table.head().round(3)

The coordinate array is `(200, 2)`. The stored graph is `(200, 200)` because its rows and columns refer to input observations. It records weighted neighbor connections in a sparse matrix. It is a different object from the two-column layout.

## Plot the Two Coordinate Arrays

The loop draws one panel per method. `coordinates[:, 0]` selects the first coordinate of every image; `coordinates[:, 1]` selects the second. Colors come from the recorded digits, in the same row order.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, coordinates, title in zip(
    axes, [tsne_coordinates, umap_coordinates], ["t-SNE", "UMAP"], strict=True
):
    scatter = ax.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        c=recorded_digits,
        cmap="tab10",
        vmin=-0.5,
        vmax=9.5,
        s=15,
    )
    ax.set(title=title, xlabel="Map coordinate 1", ylabel="Map coordinate 2")
    ax.set_aspect("equal", adjustable="datalim")
fig.colorbar(scatter, ax=axes, ticks=range(10), label="Recorded digit")
plt.show()

The panels use the same input observations and color key, but their map positions differ. Many nearby points share a recorded digit. That gives a relationship to investigate; apparent islands do not by themselves establish distinct populations. The two panels have separately fitted coordinate systems, so an x-value in one panel has no direct counterpart in the other.

## Keep the Object and Array Separate

Use array indexing when you want an existing row's map coordinates. Double brackets retain the two-dimensional shape used throughout the course.

In [ ]:
one_input = features[[0]]
one_tsne_position = tsne_coordinates[[0]]
one_umap_position = umap_coordinates[[0]]

print("one input row:", one_input.shape)
print("its stored t-SNE position:", one_tsne_position.round(3))
print("its stored UMAP position:", one_umap_position.round(3))

These are stored positions for an observation already used in fitting. Calling `fit_transform` on another dataset would fit another map. scikit-learn's `TSNE` has no `transform` method for placing new rows into its fitted map. UMAP does provide `transform` for new observations described by the same input features. Neither returned coordinate array itself supplies model methods.

## Try One Small Modification

Change `perplexity=15` to `perplexity=30`, then rerun the t-SNE fitting cell and all later cells. The output should still have 200 rows and two columns, with the same recorded digit attached to each row. Its positions can change. A changed picture alone does not establish which neighborhood setting better serves an analytical purpose.

## Ready for 05a

You are ready to return to 05a when you can recognize these patterns:

```text
features[i].reshape(8, 8)            # recover one image's known pixel grid
TSNE(...).fit_transform(features)    # fit a layout and return coordinates
UMAP(...).fit_transform(features)    # fit a graph and layout, return coordinates
model.embedding_                    # inspect stored fitted-row coordinates
pd.DataFrame(coordinates, index=...) # keep coordinates attached to observations
coordinates[:, 0]                   # select one map coordinate for all rows
```

Track what each axis refers to. Input columns describe observations; output columns locate those observations in a map. Changing a row's representation changes what the method can compare.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).